# Traffic Sign Detection — Video Test (real application demo)

Runs the trained detector on chosen parts of a driving video, frame by frame, and shows the
**current speed limit** on screen like an in-car driver-assist display.

**Before running**
1. Run `01_train_evaluate.ipynb` first — it saves `best.onnx` / `best.pt` in `weights/`.
2. Put your driving video in `videos/` and set `VIDEO_PATH` and `SEGMENTS` below.

Runs locally on CPU (~5 frames/s), so we process short segments instead of the whole video.

In [ ]:
import re
import time
from collections import Counter, deque
from pathlib import Path

import cv2
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from ultralytics import YOLO

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

VIDEO_PATH = ROOT / 'videos' / 'videoplayback.mp4'
WEIGHTS = ROOT / 'weights' / 'best.onnx'        # or ROOT / 'weights' / 'best.pt'
OUT_DIR = ROOT / 'results' / 'video'

# (start, end) as 'hh:mm:ss' — each segment is processed separately
SEGMENTS = [
    ('00:00:00', '00:00:20'),
    ('00:14:00', '00:14:20'),
    ('00:21:30', '00:21:50'),
    ('00:27:40', '00:28:00'),
]

CONF = 0.40          # minimum confidence to accept a detection
VOTE_WINDOW = 5      # recent speed-sign sightings used to decide the current limit
IMGSZ = 960          # must match the training image size (the ONNX model has a fixed input size)

for p in (VIDEO_PATH, WEIGHTS):
    assert p.exists(), f'missing {p}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def to_seconds(hms):
    h, m, s = map(float, hms.split(':'))
    return h * 3600 + m * 60 + s

assert to_seconds('00:21:30') == 1290 and to_seconds('01:00:00.5') == 3600.5

## Speed-limit logic
The detector finds signs in each frame; this small tracker turns those detections into one
stable "current speed limit". It keeps the last few speed-sign sightings and shows the most common
value, so a single wrong frame does not make the display flicker. The limit is kept until a new
speed sign is seen (just like a real road).

In [ ]:
class SpeedLimitTracker:
    def __init__(self, window=VOTE_WINDOW):
        self.recent = deque(maxlen=window)

    def update(self, labels):
        for label in labels:
            m = re.search(r'(\d+)', label) if 'speed' in label.lower() else None
            if m:
                self.recent.append(int(m.group(1)))
        return Counter(self.recent).most_common(1)[0][0] if self.recent else None

# self-check
t = SpeedLimitTracker(window=3)
assert t.update([]) is None
assert t.update(['Speed Limit 60', 'Stop']) == 60
assert t.update(['Other Sign']) == 60                          # held with no new sign
assert t.update(['Speed Limit 80']) in (60, 80)
assert t.update(['Speed Limit 80']) == 80                      # majority switches

def draw_limit(frame, limit):
    """Draw a round speed-limit sign in the top-left corner."""
    c, r = (70, 70), 50
    cv2.circle(frame, c, r, (255, 255, 255), -1)
    cv2.circle(frame, c, r, (0, 0, 220), 9)
    text = str(limit) if limit else '--'
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_DUPLEX, 1.3, 3)
    cv2.putText(frame, text, (c[0] - tw // 2, c[1] + th // 2), cv2.FONT_HERSHEY_DUPLEX, 1.3, (0, 0, 0), 3)
    return frame

## Run on each segment

In [ ]:
model = YOLO(str(WEIGHTS), task='detect')

def process_segment(start, end):
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    first, last = round(to_seconds(start) * fps), round(to_seconds(end) * fps)
    cap.set(cv2.CAP_PROP_POS_FRAMES, first)
    name = f"{start.replace(':', '')}-{end.replace(':', '')}"
    out_path = OUT_DIR / f'{name}.mp4'
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

    tracker, log, keyframes = SpeedLimitTracker(), [], []
    stride = max((last - first) // 16, 1)
    t0 = time.perf_counter()
    for i in range(first, last):
        ok, frame = cap.read()
        if not ok:
            break
        r = model.predict(frame, conf=CONF, imgsz=IMGSZ, verbose=False)[0]
        labels = [model.names[int(c)] for c in r.boxes.cls]
        limit = tracker.update(labels)
        out = draw_limit(r.plot(), limit)
        cv2.putText(out, time.strftime('%H:%M:%S', time.gmtime(i / fps)), (w - 170, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        writer.write(out)
        log.append({'time_s': i / fps, 'ms': sum(r.speed.values()), 'n_signs': len(labels),
                    'labels': labels, 'conf': [float(c) for c in r.boxes.conf], 'limit': limit})
        if labels and (i - first) % stride == 0 and len(keyframes) < 8:
            keyframes.append((i / fps, cv2.cvtColor(out, cv2.COLOR_BGR2RGB)))
    cap.release(); writer.release()
    return name, fps, pd.DataFrame(log), keyframes, time.perf_counter() - t0, out_path

runs = []
for start, end in SEGMENTS:
    print(f'processing {start} → {end} ...', end=' ')
    runs.append(process_segment(start, end))
    print(f'done in {runs[-1][4]:.0f}s → {runs[-1][5].name}')

## Performance on the video
There are no ground-truth labels for this video, so this is a **qualitative + speed** check:
processing speed, how often signs are detected, which classes appear, their confidence,
and which speed limit was displayed. Watch the annotated segments in `results/video/` to judge correctness
(e.g. count the real signs you see and how many the model caught).

In [ ]:
rows = []
for name, fps, df, _, wall, _ in runs:
    labels = [l for ls in df['labels'] for l in ls]
    conf = [c for cs in df['conf'] for c in cs]
    rows.append({
        'segment': name, 'frames': len(df), 'avg model ms/frame': round(df['ms'].mean(), 1),
        'processing FPS': round(len(df) / wall, 1), 'frames with ≥1 sign': f"{(df['n_signs'] > 0).mean():.1%}",
        'detections': len(labels), 'mean conf': round(sum(conf) / len(conf), 3) if conf else '-',
        'classes seen': ', '.join(f'{k} ×{v}' for k, v in Counter(labels).most_common()) or '-',
        'speed limits shown': ', '.join(map(str, sorted(df['limit'].dropna().astype(int).unique()))) or '-',
    })
video_summary = pd.DataFrame(rows)
display(video_summary)
video_summary.to_csv(OUT_DIR / 'video_summary.csv', index=False)
(OUT_DIR / 'video_summary.md').write_text(f'# Video test — {VIDEO_PATH.name}\n\n{video_summary.to_markdown(index=False)}\n')

In [ ]:
for name, fps, df, keyframes, _, _ in runs:
    display(Markdown(f'### Segment {name}'))
    fig, ax = plt.subplots(1, 2, figsize=(16, 3.5))
    ax[0].plot(df['time_s'], df['n_signs']); ax[0].set_title('Signs detected per frame'); ax[0].set_xlabel('video time (s)')
    ax[1].step(df['time_s'], df['limit'], where='post'); ax[1].set_title('Displayed speed limit'); ax[1].set_xlabel('video time (s)')
    ax[1].set_ylabel('km/h')
    plt.tight_layout(); plt.savefig(OUT_DIR / f'{name}_stats.png', dpi=110); plt.show()
    if not keyframes:
        print('No signs detected in this segment.')
        continue
    cols = 4
    rows_n = (len(keyframes) + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(5 * cols, 3 * rows_n), squeeze=False)
    for a in axes.flat:
        a.axis('off')
    for a, (ts, img) in zip(axes.flat, keyframes):
        a.imshow(img); a.set_title(time.strftime('%H:%M:%S', time.gmtime(ts)))
    plt.tight_layout(); plt.savefig(OUT_DIR / f'{name}_frames.png', dpi=110); plt.show()